In [1]:
import os
print(os.getcwd())
print(os.listdir())

d:\trip-planner
['data', 'day1.ipynb', 'day2.ipynb', 'day3.ipynb', 'day4.ipynb', 'day5.ipynb']


In [4]:
import pandas as pd
hotels = pd.read_csv("data/hotels.csv")
places = pd.read_csv("data/places.csv")
transport = pd.read_csv("data/transport.csv")

In [3]:
for a in ["wifi", "breakfast", "pool", "spa"]:
    hotels["has_" + a] = hotels["amenities"].str.contains(a).astype(int)

hotels.head()

,hotel_id,city,name,price_per_night,rating,category,amenities,has_wifi,has_breakfast,has_pool,has_spa
0,1,Goa,Beachside Budget Inn,1500,3.8,budget,wifi,1,0,0,0
1,2,Goa,Calangute Comfort Stay,3500,4.2,comfort,wifi | breakfast | pool,1,1,1,0
2,3,Goa,Luxury Sands Resort,9000,4.7,luxury,wifi | breakfast | pool | spa,1,1,1,1
3,4,Mysuru,Palace Road Lodge,1200,3.9,budget,wifi,1,0,0,0
4,5,Mysuru,Hotel Pai Vista,3000,4.2,comfort,wifi | breakfast | parking,1,1,0,0


In [4]:
cat = pd.get_dummies(hotels["category"], prefix="cat").astype(int)
hotels = pd.concat([hotels, cat], axis=1)

hotels.head()

,hotel_id,city,name,price_per_night,rating,category,amenities,has_wifi,has_breakfast,has_pool,has_spa,cat_budget,cat_comfort,cat_luxury
0,1,Goa,Beachside Budget Inn,1500,3.8,budget,wifi,1,0,0,0,1,0,0
1,2,Goa,Calangute Comfort Stay,3500,4.2,comfort,wifi | breakfast | pool,1,1,1,0,0,1,0
2,3,Goa,Luxury Sands Resort,9000,4.7,luxury,wifi | breakfast | pool | spa,1,1,1,1,0,0,1
3,4,Mysuru,Palace Road Lodge,1200,3.9,budget,wifi,1,0,0,0,1,0,0
4,5,Mysuru,Hotel Pai Vista,3000,4.2,comfort,wifi | breakfast | parking,1,1,0,0,0,1,0


In [5]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
hotels[["price_scaled", "rating_scaled"]] = scaler.fit_transform(
    hotels[["price_per_night", "rating"]]
)

hotels[["price_per_night", "price_scaled", "rating", "rating_scaled"]].head()

,price_per_night,price_scaled,rating,rating_scaled
0,1500,0.033333,3.8,0.1
1,3500,0.166667,4.2,0.5
2,9000,0.533333,4.7,1.0
3,1200,0.013333,3.9,0.2
4,3000,0.133333,4.2,0.5


In [6]:
feature_cols = ["price_scaled", "rating_scaled",
                "has_wifi", "has_breakfast", "has_pool", "has_spa",
                "cat_budget", "cat_comfort", "cat_luxury"]

features = hotels[feature_cols]
print(features.shape)
features.head()

(16, 9)


,price_scaled,rating_scaled,has_wifi,has_breakfast,has_pool,has_spa,cat_budget,cat_comfort,cat_luxury
0,0.033333,0.1,1,0,0,0,1,0,0
1,0.166667,0.5,1,1,1,0,0,1,0
2,0.533333,1.0,1,1,1,1,0,0,1
3,0.013333,0.2,1,0,0,0,1,0,0
4,0.133333,0.5,1,1,0,0,0,1,0


In [7]:
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(features)
print(sim.shape)

(16, 16)


In [8]:
def similar_hotels(name, n=3):
    idx = hotels.index[hotels["name"] == name][0]
    scores = sorted(enumerate(sim[idx]), key=lambda x: x[1], reverse=True)[1:n+1]
    result = hotels.iloc[[i for i, _ in scores]][
        ["name", "city", "price_per_night", "rating", "category"]
    ].copy()
    result["similarity"] = [round(s, 2) for _, s in scores]
    return result

In [9]:
similar_hotels("Beachside Budget Inn")

,name,city,price_per_night,rating,category,similarity
13,Pink City Lodge,Jaipur,1300,3.8,budget,1.0
3,Palace Road Lodge,Mysuru,1200,3.9,budget,1.0
7,Om Beach Hut,Gokarna,1000,3.7,budget,1.0
